In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, shutil, hashlib, subprocess
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)
for fn, dest in [('.git-credentials','/root/.git-credentials'), ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists(): shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


ValueError: mount failed

In [ ]:
# =============================================================================
# Cell 2 - load the Wednesday within-day environment (Amendment 7 A7.2).
# Wednesday holds DoS (four variants), Benign, and Heartbleed. This is the only
# CIC day where an attack family sits in both source and target, so it is the
# only day that can measure focal-class coverage under shift.
# =============================================================================
cic = pd.read_parquet(config.INTERIM_DIR / 'cicids2017_primary.parquet')
wed = cic[cic['day'] == 'wednesday'].reset_index(drop=True)

print('Wednesday rows:', len(wed))
print('\nfamilies on Wednesday:')
print(wed['label'].value_counts().to_string())
print('\nDoS variants on Wednesday:')
print(wed[wed['label'] == 'DoS']['subtype'].value_counts().to_string())

FOCAL_DAY = 'wednesday'
FEATURE_EXCLUDE = ['label_raw','label','subtype','day','source_file','attempted_relabelled_benign']


In [ ]:
# =============================================================================
# Cell 3 - capture-time signal for the session split (Amendment 7 A7.2).
# Prefer a real timestamp column. If the cleaned parquet dropped it, fall back
# to within-file flow order as the capture-order proxy. Record which was used;
# the choice bounds how clean the W-cov covariate shift is.
# =============================================================================
time_col = None
for c in wed.columns:
    lc = c.lower()
    if 'timestamp' in lc or lc == 'time' or ('time' in lc and 'flow' not in lc and 'iat' not in lc and 'active' not in lc and 'idle' not in lc):
        parsed = pd.to_datetime(wed[c], errors='coerce')
        if parsed.notna().mean() > 0.95:
            time_col = c
            wed['_capture_ts'] = parsed
            break

if time_col is not None:
    order = wed['_capture_ts'].rank(method='first').to_numpy()
    SPLIT_METHOD = f'timestamp column "{time_col}"'
else:
    order = np.arange(len(wed))            # parquet preserves flow order from the pcap
    SPLIT_METHOD = 'within-file flow order (no usable timestamp column found)'

wed['_order'] = order
print('columns containing time-like names:',
      [c for c in wed.columns if 'time' in c.lower()])
print('session split will use:', SPLIT_METHOD)
if time_col is None:
    print('NOTE: flow-order proxy is weaker than real time. W-cov S_cov will still be '
          'measured empirically per section 9; if it is near the permutation null, W-cov '
          'is reported as low-shift rather than assumed.')


In [ ]:
# =============================================================================
# Cell 4 - canonical W-cov session split: earlier half = source, later half =
# target, ordered by capture time. This is the BINDING reference split used for
# feasibility and the focal class. Notebook 11 generates the R=5 realizations
# around it (varied split points) for shift variation.
# =============================================================================
PARTITION_SEED_CIC = 20260724

cut = np.median(wed['_order'])
wed['_block'] = np.where(wed['_order'] <= cut, 'source', 'target')

src_block = wed[wed['_block'] == 'source'].copy()
tgt_block = wed[wed['_block'] == 'target'].copy()

print(f'source block: {len(src_block)}   target block: {len(tgt_block)}')
comp = pd.DataFrame({
    'source_block': src_block['label'].value_counts(),
    'target_block': tgt_block['label'].value_counts(),
}).fillna(0).astype(int)
print('\nclass composition per block:')
print(comp.to_string())
print('\nDoS present in both blocks:',
      (comp.loc['DoS'] > 0).all() if 'DoS' in comp.index else False)


In [ ]:
# =============================================================================
# Cell 5 - five-partition split of the W-cov SOURCE block (preregistration 4).
# 60/10/15/15 into train, val, probcal, source_cal_pool, stratified by family.
# D_eval and T_cal are drawn from the TARGET block in notebook 11.
# =============================================================================
def stratified_split(df, fractions, seed, stratify_col='label'):
    rng = np.random.default_rng(seed)
    names = list(fractions); fracs = np.array([fractions[k] for k in names], float)
    assert abs(fracs.sum() - 1.0) < 1e-9
    largest = names[int(np.argmax(fracs))]
    assign = pd.Series(index=df.index, dtype=object)
    for cls, sub in df.groupby(stratify_col, sort=True):
        idx = sub.index.to_numpy().copy(); rng.shuffle(idx); n = len(idx)
        counts = np.floor(fracs * n).astype(int); counts[names.index(largest)] += n - counts.sum()
        start = 0
        for name, c in zip(names, counts):
            assign.loc[idx[start:start + c]] = name; start += c
    return assign

src_block = src_block.assign(partition=stratified_split(src_block, config.SPLIT_FRACTIONS, PARTITION_SEED_CIC).values)

parts = {k: src_block[src_block.partition == k] for k in config.SPLIT_FRACTIONS}
assert sum(len(p) for p in parts.values()) == len(src_block), 'row loss in split'
S_pool = parts['source_cal_pool']

for name, p in parts.items():
    print(f'{name:16s} {len(p):8d}  {len(p)/len(src_block):6.3f}')
print(f'{"source block":16s} {len(src_block):8d}')


In [ ]:
# =============================================================================
# Cell 6 - BINDING feasibility on the Wednesday source calibration pool.
# A Wednesday family supports class-conditional (Mondrian) conformal at level
# alpha only if its count in S_pool satisfies n_c >= ceil(1/alpha) - 1
# (preregistration 7.6). Infeasible families are excluded, not forced.
# =============================================================================
alphas = [config.ALPHA_PRIMARY] + config.ALPHA_SENSITIVITY + config.ALPHA_CONDITIONAL
scal = S_pool['label'].value_counts()

rows = []
for a in alphas:
    need = config.min_calib_n(a)
    for cls in scal.index:
        n = int(scal[cls])
        rows.append({'dataset':'cicids2017','environment':'wednesday_within_day',
                     'alpha':a,'class':cls,'source_calib_n':n,
                     'min_calib_needed':need,'feasible': n >= need})
feas = pd.DataFrame(rows)
feas.to_csv(config.REPORTS_DIR / 'feasibility_binding_cicids2017.csv', index=False)
print(feas[feas.alpha == config.ALPHA_PRIMARY].to_string(index=False))


In [ ]:
# =============================================================================
# Cell 7 - focal class, BINDING (preregistration 12; Amendment 7 A7.4).
# Rarest attack family satisfying 7.6 at the primary alpha in the Wednesday
# source calibration pool. Written once, to a CIC-specific file so the NSL-KDD
# focal record is untouched. Never revised after any coverage exists.
# =============================================================================
attacks = [c for c in scal.index if c != 'Benign']
elig = feas[(feas.alpha == config.ALPHA_PRIMARY) & (feas.feasible) &
            (feas['class'].isin(attacks))].sort_values('source_calib_n')
assert len(elig) > 0, 'no feasible focal class on Wednesday'
FOCAL_CLASS = str(elig.iloc[0]['class'])

excluded = feas[(feas.alpha == config.ALPHA_PRIMARY) & (~feas.feasible) &
                (feas['class'].isin(attacks))]['class'].tolist()

record = {
    'dataset':'cicids2017','environment':'wednesday_within_day',
    'focal_class':FOCAL_CLASS,'alpha_primary':config.ALPHA_PRIMARY,
    'rule':'rarest attack family satisfying 7.6 at primary alpha in the Wednesday source calibration pool',
    'source_calib_counts':{c:int(scal[c]) for c in scal.index},
    'min_calib_needed':config.min_calib_n(config.ALPHA_PRIMARY),
    'excluded_attack_families':excluded,
    'partition_seed':PARTITION_SEED_CIC,'session_split_method':SPLIT_METHOD,
    'status':'BINDING - no CIC coverage number has been computed at time of writing',
}
fp = config.REPORTS_DIR / 'focal_class_record_cicids2017.json'
if fp.exists():
    prior = json.loads(fp.read_text())
    if prior.get('focal_class') != FOCAL_CLASS:
        raise RuntimeError(f"focal class was {prior.get('focal_class')}, now {FOCAL_CLASS}. Do not overwrite; log a deviation.")
    print('focal class already recorded and unchanged')
else:
    fp.write_text(json.dumps(record, indent=2)); print('focal class RECORDED')
print(json.dumps(record, indent=2))


In [ ]:
# =============================================================================
# Cell 8 - DoS variant inventory for the W-sup novel-variant ladder
# (Amendment 7 A7.2). Records variant counts in the source pool and the target
# block so notebook 11 can build R=5 held-out-variant realizations. Recorded,
# not yet realized.
# =============================================================================
dos_src = S_pool[S_pool['label'] == 'DoS']['subtype'].value_counts()
dos_tgt = tgt_block[tgt_block['label'] == 'DoS']['subtype'].value_counts()
var_tbl = pd.DataFrame({'source_pool': dos_src, 'target_block': dos_tgt}).fillna(0).astype(int)
var_tbl.index.name = 'dos_variant'
var_tbl.to_csv(config.REPORTS_DIR / 'cicids2017_dos_variants.csv')
print('DoS variants available for the W-sup holdout:')
print(var_tbl.to_string())
print('\nn variants:', len(var_tbl),
      '| min for a held-out-vs-kept split:', 2)


In [ ]:
# =============================================================================
# Cell 9 - partition fingerprints + commit. Index arrays go to data/ (gitignored);
# fingerprints and tables to reports/ (committed) so the split is verifiable
# without the data.
# =============================================================================
config.PROC_DIR.mkdir(parents=True, exist_ok=True)
def fingerprint(idx):
    return hashlib.sha256(np.sort(np.asarray(idx, np.int64)).tobytes()).hexdigest()

fps = {'session_block_source': fingerprint(src_block.index),
       'session_block_target': fingerprint(tgt_block.index)}
for name, p in parts.items():
    np.save(config.PROC_DIR / f'cicids2017_wed_source_{name}_idx.npy', np.sort(p.index.to_numpy()))
    fps[f'source/{name}'] = {'n': int(len(p)), 'sha256': fingerprint(p.index)}

(config.REPORTS_DIR / 'partition_fingerprints_cicids2017.json').write_text(json.dumps(
    {'partition_seed':PARTITION_SEED_CIC,'environment':'wednesday_within_day',
     'session_split_method':SPLIT_METHOD,'split_fractions':config.SPLIT_FRACTIONS,
     'fingerprints':fps}, indent=2))

# fold in the deviation line that missed the earlier commit
dev = config.REPORTS_DIR / 'deviations.md'
note = '\n## nb10 note: amendments 7-8 committed within 5b5cfc5 (nb09 message); provenance intact, before any coverage.\n'
if dev.exists() and 'amendments 7-8 committed within 5b5cfc5' not in dev.read_text():
    with open(dev, 'a') as f: f.write(note)

def git(*a, show=True):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    if show and (r.stdout or r.stderr): print((r.stdout + r.stderr).strip())
    return r
for s, d in [('/root/.git-credentials', PARENT_DIR/'.git-credentials'), ('/root/.gitconfig', PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, d)
os.chdir(PROJECT_ROOT)
git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb10: CIC Wednesday within-day partitions, feasibility, focal class (DoS)')
    r = git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-4', show=False).stdout)
